# Extending **MesoMath**

## The "Vertical Problem": Defining Height

In many Mesopotamian mathematical problems, vertical measurements (height or depth) are treated with 
specific units that, while sharing the same names as lengths, behave differently in calculations.

Let's define a custom class `bh` (Babylonian Height) that inherits from the length system 
(`bl` or `Blen`) but fixes the base unit to the *kuš3* (cubit).

In [1]:
from mesomath.npvs import Blen, Bsur, Bvol, Bcap

# We define our custom height class
class bh(Blen):
    title: str = "Babylonian Height Measurement"
    ubase: int = 1  # Fixed to 'kus' (cubit)

## Seamless Interaction

Thanks to the polymorphic design of **MesoMath**, your custom classes are recognized by the core engine. 
    You can multiply a standard surface (`Bsur`) by your new height (`bh`) to obtain a volume (`Bvol`) 
without any type errors.

In [2]:
# 1. Define a surface of 1 sar
area = Bsur('1 sar')

# 2. Define a height using our custom class
height = bh('1 kus')

# 3. Calculate volume (Area * Height)
# The system recognizes 'bh' as a valid length for this operation.
volume = area * height

print(f"Volume: {volume}") 
# Output: 1 sar

Volume: 1 sar


## Automatic Utilities

By inheriting from `MesoM` (via `Blen`), your new class automatically gains all the new administrative and diagnostic tools of this version:

1. **Metrological Lists/Tables**: Generate tables for your custom units instantly.
   

In [3]:
bh.metrolist('1 kus', '5 kus', '1 kus', verbose=True, ubase=None)

Measurement          | Sexag. (base)
--------------------------------------
1 kus                | 1              
2 kus                | 2              
3 kus                | 3              
4 kus                | 4              
5 kus                | 5              


In [4]:
bh.metrolist('10 susi', '2 kus', '5 susi', verbose=True, width=30,fractions=2,actual=True)

Measurement                    | Sexag. (base)
------------------------------------------------
1/3 kuš3                       | 20             
1/2 kuš3                       | 30             
2/3 kuš3                       | 40             
5/6 kuš3                       | 50             
1 kuš3                         | 1              
1 1/6 kuš3                     | 1:10           
1 1/3 kuš3                     | 1:20           
1 1/2 kuš3                     | 1:30           
1 2/3 kuš3                     | 1:40           
1 5/6 kuš3                     | 1:50           
1/6 ninda                      | 2              


In [5]:
           
bh.prtsex = 1
bh.metrolist('10 susi', '2 kus', '5 susi', verbose=True, width=30,fractions=2,actual=True)

Measurement                    | Sexag. (base)
------------------------------------------------
1/3 kuš3                       | 20             
1/2 kuš3                       | 30             
2/3 kuš3                       | 40             
5/6 kuš3                       | 50             
(1 dis) kuš3                   | 1              
(1 dis) 1/6 kuš3               | 1:10           
(1 dis) 1/3 kuš3               | 1:20           
(1 dis) 1/2 kuš3               | 1:30           
(1 dis) 2/3 kuš3               | 1:40           
(1 dis) 5/6 kuš3               | 1:50           
1/6 ninda                      | 2              


The following are the options for the `.metrolist()` method:

In [6]:
help(bh.metrolist)

Help on method metrolist in module mesomath.npvs:

metrolist(mmin: str | int, mmax: str | int, step: str | int, verbose: bool = False, ubase: int | None = None, width: int = 20, fractions: int = -1, actual: bool = False, echo: bool = True, **kwargs) class method of __main__.bh
    Generate a list of metrological values for the current class.

    :param mmin: Initial value (e.g., '1 ninda' or integer)
    :type mmin: str | int
    :param mmax: Final value
    :type mmax: str | int
    :param step: Increment
    :type step: str | int
    :param verbose: If it is True, it returns the floating metrological value, defaults to False
    :type verbose: bool, optional
    :param ubase: force ubase unit, defaults to None
    :type ubase: int, optional
    :param width: output width, defaults to 20
    :type width: int, optional
    :param fractions: Use fractions if 1 and add 1/6 if 2, defaults to -1 (no fractions)
    :type fractions: int, optional
    :param actual: use academic unit names
 

## Late Babylonian Period Metrology


**MesoMath** is designed to work with the metrology of the Old Babylonian period, but it can be extended to use the metrology of other periods. For example, for the Late Babylonian Period, we can start by defining a class `LBcap` for the capacities:

In [7]:
class LBcap(Bcap):  # Capacity
    """This class implement Non-Place-Value System arithmetic
    for Late Babylonian Period capacity units:

        **gur <-5- bariga <-6- ban2 <-10- sila3 <-10- GAR**

    """

    title: str = "Late Babylonian capacity meassurement"
    uname: list[str] = "gar sila ban bariga gur".split()
    aname: list[str] = "GAR sila3 ban2 bariga gur".split()
    ufact: list[int] = [10, 10, 6, 5]
    cfact: list[int] = [1, 10, 100, 600, 3000]
    siv: float = 0.1
    siu: str = "litres"
    ubase: int = 3  # bariga

    def vol(self) -> object:
        """Convert capacity to volume meassurement

        :return: volume meassurement
        :rtype: "Bvol"
        """
        return LBvol(int(round(self.dec/(100/6))))

class LBvol(Bvol):  # Volume
    """This class implement Non-Place-Value System arithmetic
    for Late Babylonian Period volume units:

        **GAN2 <-100- sar <-60- gin2 <-180- še**

    """
    title: str = "Late Babylonian volume meassurement"
    
    def cap(self) -> object:
        """Convert volume to capacity meassurement"""
        return LBcap(int(round(self.dec*(100/6))))

and then:

In [8]:
a = LBcap('1000 sila')
b = a.vol()
print(f"{b =}")

b =3 gin 60 se


In [9]:
b.explain() 

This is a Late Babylonian volume meassurement: 3 gin 60 se
    Metrology:  gan <-100- sar <-60- gin <-180- se
    Factor with unit 'se':  1 180 10800 1080000
Meassurement in terms of the smallest unit: 600 (se)
Sexagesimal floating value of the above: 10
Approximate SI value: 0.9999999999999999 cube meters


In [10]:
c = b.cap() 
print(f"{c =}")

c =3 gur 1 bariga 4 ban


In [11]:
c.SI() 

'1000.0 litres'

In [12]:
c.explain() 

This is a Late Babylonian capacity meassurement: 3 gur 1 bariga 4 ban
    Metrology:  gur <-5- bariga <-6- ban <-10- sila <-10- gar
    Factor with unit 'gar':  1 10 100 600 3000
Meassurement in terms of the smallest unit: 10000 (gar)
Sexagesimal floating value of the above: 2:46:40
Approximate SI value: 1000.0 litres


In [13]:
LBcap.metrolist('1 bariga','3 bariga', '1 ban',1)

Measurement          | Sexag. (base)
--------------------------------------
1 bariga             | 1              
1 bariga 1 ban       | 1:10           
1 bariga 2 ban       | 1:20           
1 bariga 3 ban       | 1:30           
1 bariga 4 ban       | 1:40           
1 bariga 5 ban       | 1:50           
2 bariga             | 2              
2 bariga 1 ban       | 2:10           
2 bariga 2 ban       | 2:20           
2 bariga 3 ban       | 2:30           
2 bariga 4 ban       | 2:40           
2 bariga 5 ban       | 2:50           
3 bariga             | 3              


etc. but we should also redefine the rest of the classes to ensure consistency in the operations with the new units.